<a href="https://colab.research.google.com/github/KanthiPhoosorn/CareMind/blob/feature%2Fpatient-staff-chatbots/CareMind_Custom_Citation_Based_Medical_Chatbot_for_Patient.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CareMind — Custom Citation-Based Medical Chatbot for Patient (No External LLMs)

This notebook demonstrates how to build a **fully local, custom medical chatbot system** for CareMind using ONLY:

- Your own hospital datasets
- Retrieval pipelines
- Rule-based reasoning
- Citation generation
- TF-IDF semantic matching
- No OpenAI
- No Gemini
- No Qwen
- No Ollama
- No hosted APIs

---

# Important Clarification

This notebook does **NOT** create a frontier large language model from scratch.

Instead, it builds:

```text
Medical Retrieval + Citation Engine
+ Workflow Intelligence
+ Rule-Based Clinical Responses
+ Search-Based Patient Assistant
```

This is the correct MVP architecture for CareMind.

---

# Why This Approach Matters

Training a true medical LLM from scratch requires:

- billions of tokens
- massive GPU clusters
- months of training
- extremely large budgets

For CareMind MVP, the correct architecture is:

```text
Structured Retrieval
+ Citation Engine
+ Medical Rules
+ Search Intelligence
```

This produces:
- safer outputs
- deterministic citations
- better PHI control
- easier hospital deployment


In [1]:

# Core imports

import os
import re
import glob
import pandas as pd

from dataclasses import dataclass
from typing import List, Dict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



# Step 1 — Load Hospital Data

This notebook loads the provided `data.zip` structure.

Expected layout:

```text
data/
  AN1/
    DoctorProgress_note.xlsx
    NURSE_Note.xlsx
    order (drug).xlsx
    order (lab).xlsx
    order (xray).xlsx
```


In [10]:
import os
import zipfile

zip_path = '/content/data.zip'
target_path = '/content/caremind_data'

# Create the directory if it doesn't exist
os.makedirs(target_path, exist_ok=True)

# Unzip the data
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(target_path)
    print(f'Successfully extracted {zip_path} to {target_path}')
else:
    print(f'Error: {zip_path} not found.')

Successfully extracted /content/data.zip to /content/caremind_data


In [9]:
import os
print(f'Current Working Directory: {os.getcwd()}')
print('Files in /content:')
print(os.listdir('/content')) if os.path.exists('/content') else print('/content not found')

Current Working Directory: /content
Files in /content:
['.config', 'data.zip', 'sample_data']


In [11]:
# Check the extracted directory structure
target_path = '/content/caremind_data'
for root, dirs, files in os.walk(target_path):
    level = root.replace(target_path, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files[:3]: # Show first 3 files
        print(f'{subindent}{f}')

caremind_data/
    data/
        AN2/
            AN2_NURSE_Note.xlsx
            AN2_order (lab).xlsx
            AN2_DoctorProgress_note.xlsx
        AN9/
            AN9_order (drug).xlsx
            AN9_NURSE_Note.xlsx
            AN9_DoctorProgress_note.xlsx
        AN4/
            AN4_order (lab).xlsx
            AN4_order (drug).xlsx
            AN4_NURSE_Note.xlsx
        AN8/
            AN8_NURSE_Note.xlsx
            AN8_order (drug).xlsx
            AN8_order (xray).xlsx
        AN6/
            AN6_NURSE_Note.xlsx
            AN6_order (xray).xlsx
            AN6_order (lab).xlsx
        AN3/
            AN3_order (xray).xlsx
            AN3_order (lab).xlsx
            AN3_NURSE_Note.xlsx
        AN7/
            AN7_DoctorProgress_note.xlsx
            AN7_NURSE_Note.xlsx
            AN7_order (xray).xlsx
        AN5/
            AN5_NURSE_Note.xlsx
            AN5_order (xray).xlsx
            AN5_DoctorProgress_note.xlsx
        AN1/
            AN1_order (xray).xlsx


In [12]:
# Find all Excel files in the correctly identified path
target_path = '/content/caremind_data'
excel_files = glob.glob(
    os.path.join(target_path, "**", "*.xlsx"),
    recursive=True
)

print(f"Found {len(excel_files)} clinical data files.")

Found 48 clinical data files.


In [5]:

excel_files[:10]


[]


# Step 2 — Convert Clinical Files into Retrieval Chunks

Each row becomes a searchable citation chunk.

This is the foundation of citation-based AI.


In [13]:
@dataclass
class ClinicalChunk:
    chunk_id: str
    patient_an: str
    source_file: str
    row_index: int
    text: str

chunks: List[ClinicalChunk] = []
chunk_counter = 0

for file_path in excel_files:
    try:
        # Load the excel file
        df = pd.read_excel(file_path)

        # Extract the patient ID (AN) from the folder name
        patient_an = os.path.basename(os.path.dirname(file_path))
        source_file = os.path.basename(file_path)

        for idx, row in df.iterrows():
            # Filter out empty values and join row data into a single searchable string
            values = [str(v) for v in row.values if pd.notna(v)]
            combined_text = " | ".join(values).strip()

            if len(combined_text) < 5:
                continue

            chunks.append(ClinicalChunk(
                chunk_id=f"chunk_{chunk_counter}",
                patient_an=patient_an,
                source_file=source_file,
                row_index=idx,
                text=combined_text
            ))
            chunk_counter += 1
    except Exception as e:
        print(f"Failed to process: {file_path} due to {e}")

print(f"Successfully created {len(chunks)} searchable clinical chunks.")

Successfully created 9723 searchable clinical chunks.


In [14]:
# Rebuild the dataframe and vectorizer
chunk_df = pd.DataFrame([vars(x) for x in chunks])

if not chunk_df.empty:
    vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
    X = vectorizer.fit_transform(chunk_df["text"])
    print(f"Search index built with shape: {X.shape}")
else:
    print("Warning: No data chunks were created.")

Search index built with shape: (9723, 6746)



# Step 3 — Build Local Semantic Search

Instead of embeddings from external models, this notebook uses:

- TF-IDF vectors
- cosine similarity

Advantages:
- fully offline
- deterministic
- lightweight
- explainable
- CPU-friendly


In [15]:
# Re-initialize vectorizer on the full dataset to ensure consistency
vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
X = vectorizer.fit_transform(chunk_df["text"])
print(f"Vector matrix shape: {X.shape}")

Vector matrix shape: (9723, 6746)



# Step 4 — Local Retrieval Engine

This performs:
- semantic search
- patient filtering
- citation retrieval


In [16]:
def retrieve_chunks(query: str, patient_an: str, top_k: int = 5):
    scoped_df = chunk_df[chunk_df["patient_an"] == patient_an].copy()
    if scoped_df.empty:
        return pd.DataFrame()

    scoped_indices = scoped_df.index.tolist()
    scoped_matrix = X[scoped_indices]
    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(query_vector, scoped_matrix)[0]
    scoped_df["score"] = similarities
    scoped_df = scoped_df.sort_values(by="score", ascending=False)

    return scoped_df.head(top_k)

In [17]:
# Test retrieval
test_results = retrieve_chunks(query="fever and cough", patient_an="AN1")
display(test_results)

,chunk_id,patient_an,source_file,row_index,text,score
7932,chunk_7932,AN1,AN1_order (xray).xlsx,0,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0
7933,chunk_7933,AN1,AN1_order (xray).xlsx,1,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0
7934,chunk_7934,AN1,AN1_order (xray).xlsx,2,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0
7935,chunk_7935,AN1,AN1_NURSE_Note.xlsx,0,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0
7936,chunk_7936,AN1,AN1_NURSE_Note.xlsx,1,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0


# Step 5 — Intent Detection

The chatbot should understand:
- symptom screening
- queue guidance
- medication questions
- appointment help
- FAQ

In [20]:
def detect_intent(query: str):
    q = query.lower()
    if any(x in q for x in ["wait", "queue", "appointment"]):
        return "workflow"
    if any(x in q for x in ["drug", "medicine", "medication"]):
        return "medication"
    if any(x in q for x in ["fever", "pain", "cough", "headache"]):
        return "symptom"
    return "faq"


# Step 6 — PHI Redaction

Before generation:
- remove AN
- remove HN
- remove identifiers


In [21]:
def redact_phi(text: str):
    text = re.sub(r"AN\d+", "[REDACTED_AN]", text)
    text = re.sub(r"HN\d+", "[REDACTED_HN]", text)
    return text


# Step 7 — Citation Builder

Every answer must include:
- source file
- row reference
- patient scope


In [22]:
def build_citations(results: pd.DataFrame):
    citations = []
    for _, row in results.iterrows():
        citation = f"{row['source_file']} (row {row['row_index']})"
        citations.append(citation)
    return citations


# Step 8 — Build a Custom Medical Chatbot

This is NOT an LLM.

It is:
- retrieval-based
- citation-grounded
- deterministic
- explainable
- hospital-safe


In [23]:
class CareMindMedicalChatbot:
    def answer(self, query: str, patient_an: str):
        intent = detect_intent(query)
        retrieved = retrieve_chunks(query=query, patient_an=patient_an)
        citations = build_citations(retrieved)
        if retrieved.empty:
            return {
                "intent": intent,
                "answer": "I could not find enough information to answer safely.",
                "citations": []
            }
        top_text = redact_phi(retrieved.iloc[0]["text"])
        if intent == "symptom":
            answer = f"Your records contain symptom-related information. Please consult hospital staff if symptoms worsen. Related record: {top_text[:250]}"
        elif intent == "medication":
            answer = "Medication-related records were found. Please verify medication usage with clinical staff."
        elif intent == "workflow":
            answer = "Workflow or appointment-related records were found."
        else:
            answer = "Relevant hospital records were found."
        return {
            "intent": intent,
            "answer": answer,
            "citations": citations
        }


# Step 9 — Test the Chatbot


In [24]:
# Final Test of the Chatbot
bot = CareMindMedicalChatbot()
response = bot.answer(query="fever and cough", patient_an="AN1")
print(response)

{'intent': 'symptom', 'answer': 'Your records contain symptom-related information. Please consult hospital staff if symptoms worsen. Related record: [REDACTED_AN] | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:02:23 | 20-DEC-2025 | 03:42:52 | E.C.G. (Electrocardiography)', 'citations': ['AN1_order (xray).xlsx (row 0)', 'AN1_order (xray).xlsx (row 1)', 'AN1_order (xray).xlsx (row 2)', 'AN1_NURSE_Note.xlsx (row 0)', 'AN1_NURSE_Note.xlsx (row 1)']}


### Comprehensive Chatbot Testing
We will now test the system across different patient records and clinical intents.

In [25]:
test_scenarios = [
    {"query": "What medications are prescribed?", "an": "AN2"},
    {"query": "When is the next appointment or queue status?", "an": "AN10"},
    {"query": "Patient has a severe headache", "an": "AN3"},
    {"query": "Are there any lab results for glucose?", "an": "AN10"}
]

for scenario in test_scenarios:
    print(f"--- Testing Query: '{scenario['query']}' for Patient: {scenario['an']} ---")
    res = bot.answer(query=scenario['query'], patient_an=scenario['an'])
    print(res)
    print("\n")

--- Testing Query: 'What medications are prescribed?' for Patient: AN2 ---
{'intent': 'medication', 'answer': 'Medication-related records were found. Please verify medication usage with clinical staff.', 'citations': ['AN2_NURSE_Note.xlsx (row 0)', 'AN2_NURSE_Note.xlsx (row 1)', 'AN2_NURSE_Note.xlsx (row 2)', 'AN2_NURSE_Note.xlsx (row 3)', 'AN2_NURSE_Note.xlsx (row 4)']}


--- Testing Query: 'When is the next appointment or queue status?' for Patient: AN10 ---
{'intent': 'workflow', 'answer': 'Workflow or appointment-related records were found.', 'citations': ['AN10_order (lab).xlsx (row 275)', 'AN10_DoctorProgress_note.xlsx (row 8)', 'AN10_DoctorProgress_note.xlsx (row 39)', 'AN10_DoctorProgress_note.xlsx (row 0)', 'AN10_DoctorProgress_note.xlsx (row 2)']}


--- Testing Query: 'Patient has a severe headache' for Patient: AN3 ---
{'intent': 'symptom', 'answer': 'Your records contain symptom-related information. Please consult hospital staff if symptoms worsen. Related record: [REDACTED


# Step 10 — What Makes This Safer Than Generic LLMs

This architecture:
- never invents unsupported records
- only answers from retrieved evidence
- includes citations
- keeps PHI local
- avoids external APIs

---

# Step 11 — Future Upgrade Path

You can gradually evolve this into a true medical language model stack.

## Phase 1 (Current)
- TF-IDF retrieval
- deterministic responses
- citations
- rules

## Phase 2
Build your own embeddings:
- train SentencePiece tokenizer
- train clinical embeddings on Thai hospital data
- build vector search

## Phase 3
Train your own transformer:
- PyTorch
- decoder-only architecture
- clinical corpus pretraining
- instruction tuning
- retrieval grounding

## Phase 4
Fine-tune:
- Thai medical language
- queue workflows
- symptom triage
- hospital FAQ
- discharge instructions

---

# IMPORTANT

Do NOT start by training a giant LLM.

The correct order is:

```text
Data Cleaning
→ Retrieval
→ Citation Engine
→ Security
→ Evaluation
→ Embeddings
→ Small Transformer
→ Larger Models
```

For hospital systems:
- retrieval quality matters more than model size
- citations matter more than fluent text
- safety matters more than creativity



# Phase 2 — Custom Embedding System

Phase 1 used TF-IDF retrieval.

Phase 2 upgrades CareMind into a true semantic medical retrieval system.

Goals:
- train local embeddings
- improve Thai clinical search
- improve symptom matching
- support multilingual retrieval
- improve citation quality

---

# Recommended Architecture

```text
Clinical Text
    ↓
Tokenizer
    ↓
Embedding Model
    ↓
Vector Database
    ↓
Semantic Retrieval
    ↓
Citation Validation
```


In [31]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
import numpy as np
print("Phase 2 libraries imported.")

Phase 2 libraries imported.



# Step 1 — Build Dense Semantic Embeddings

Instead of sparse TF-IDF only,
we create compressed semantic vectors.

This simulates a lightweight embedding model.


In [32]:
# Create dense embeddings from TF-IDF
svd = TruncatedSVD(n_components=128, random_state=42)
dense_embeddings = svd.fit_transform(X)
dense_embeddings = normalize(dense_embeddings)
print(f"Dense embeddings created with shape: {dense_embeddings.shape}")

Dense embeddings created with shape: (9723, 128)



# Step 2 — Semantic Vector Retrieval

This is much closer to modern retrieval systems.


In [27]:
def semantic_retrieve(query: str, patient_an: str, top_k: int = 5):
    scoped_df = chunk_df[chunk_df["patient_an"] == patient_an].copy()
    if scoped_df.empty:
        return pd.DataFrame()
    scoped_indices = scoped_df.index.tolist()
    scoped_vectors = dense_embeddings[scoped_indices]
    query_tfidf = vectorizer.transform([query])
    query_dense = svd.transform(query_tfidf)
    query_dense = normalize(query_dense)
    # Calculate cosine similarity via dot product
    scores = np.dot(scoped_vectors, query_dense.T).flatten()
    scoped_df["semantic_score"] = scores
    scoped_df = scoped_df.sort_values(by="semantic_score", ascending=False)
    return scoped_df.head(top_k)

In [33]:
# Test Phase 2 Semantic Retrieval
semantic_results = semantic_retrieve(query="high fever and sore throat", patient_an="AN1")
display(semantic_results)

,chunk_id,patient_an,source_file,row_index,text,semantic_score
7942,chunk_7942,AN1,AN1_NURSE_Note.xlsx,7,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.195447
8033,chunk_8033,AN1,AN1_DoctorProgress_note.xlsx,0,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.156676
8034,chunk_8034,AN1,AN1_DoctorProgress_note.xlsx,1,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.123761
8036,chunk_8036,AN1,AN1_DoctorProgress_note.xlsx,3,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.108553
7974,chunk_7974,AN1,AN1_NURSE_Note.xlsx,39,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.106854



# Step 3 — Hybrid Retrieval

Best practice in healthcare retrieval:

```text
Keyword Search
+ Semantic Search
+ Metadata Filters
```

This improves:
- citation quality
- recall
- symptom matching
- medication lookup


In [29]:
def hybrid_retrieve(query: str, patient_an: str, top_k: int = 5):
    # Keyword (TF-IDF)
    keyword_results = retrieve_chunks(query=query, patient_an=patient_an, top_k=top_k)
    # Semantic (SVD)
    semantic_results = semantic_retrieve(query=query, patient_an=patient_an, top_k=top_k)
    combined = pd.concat([keyword_results, semantic_results]).drop_duplicates(subset=["chunk_id"])
    # Combine scores
    score_columns = [c for c in combined.columns if "score" in c]
    combined["final_score"] = combined[score_columns].fillna(0).sum(axis=1)
    return combined.sort_values(by="final_score", ascending=False).head(top_k)

In [34]:
# Final Hybrid Test
hybrid_results = hybrid_retrieve(query="medication for fever", patient_an="AN1")
display(hybrid_results)

,chunk_id,patient_an,source_file,row_index,text,score,semantic_score,final_score
8035,chunk_8035,AN1,AN1_DoctorProgress_note.xlsx,2,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,NaN,0.262371,0.262371
8034,chunk_8034,AN1,AN1_DoctorProgress_note.xlsx,1,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,NaN,0.258372,0.258372
8033,chunk_8033,AN1,AN1_DoctorProgress_note.xlsx,0,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,NaN,0.256375,0.256375
8036,chunk_8036,AN1,AN1_DoctorProgress_note.xlsx,3,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,NaN,0.231984,0.231984
7999,chunk_7999,AN1,AN1_NURSE_Note.xlsx,64,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,NaN,0.141658,0.141658



# Phase 3 — Build Your Own Small Transformer

This phase moves toward a real custom language model.

DO NOT attempt a giant GPT-scale model.

Instead:
- train a small clinical transformer
- focus on Thai medical data
- optimize for retrieval + summarization

---

# Recommended Initial Specs

| Component | Recommendation |
|---|---|
| Framework | PyTorch |
| Parameters | 20M–150M |
| Tokenizer | SentencePiece |
| Context Window | 2048 |
| Objective | Next-token prediction |
| Hardware | Single GPU initially |

---

# Training Corpus

Use:
- doctor notes
- nurse notes
- medication orders
- discharge summaries
- queue records
- FAQ workflows

Do NOT train on:
- unrestricted internet data
- random Reddit dumps
- unverified medical text


In [ ]:

# Example transformer tokenizer training plan

training_plan = {
    "phase": "small_transformer",
    "tokenizer": "SentencePiece",
    "vocab_size": 32000,
    "target_parameters": "50M",
    "training_objective": "causal_language_modeling",
    "primary_data": [
        "doctor_notes",
        "nurse_notes",
        "lab_orders",
        "queue_workflows"
    ]
}

training_plan



# Phase 4 — Clinical Safety Layer

Even with your own model:

The model is NEVER trusted directly.

You still need:
- retrieval grounding
- citations
- schema validation
- hallucination detection
- audit logs


In [ ]:

def validate_citations(
    answer: str,
    citations: list
):

    if len(citations) == 0:
        return False

    if len(answer.strip()) < 10:
        return False

    return True



# Phase 5 — Thai Medical Language Optimization

Real Thai hospitals contain:
- Thai + English mixing
- abbreviations
- shorthand
- inconsistent formatting

You will eventually need:
- Thai medical normalization
- custom tokenization
- abbreviation expansion
- multilingual embeddings


In [ ]:

THAI_MEDICAL_ABBREVIATIONS = {
    "HT": "hypertension",
    "DM": "diabetes mellitus",
    "SOB": "shortness of breath",
    "AF": "atrial fibrillation"
}

THAI_MEDICAL_ABBREVIATIONS



# Phase 6 — Production Architecture

Final CareMind AI stack:

```text
Frontend
    ↓
AI Gateway
    ↓
Auth + RLS
    ↓
Hybrid Retrieval
    ↓
Citation Engine
    ↓
Custom Transformer
    ↓
Safety Validation
    ↓
Streaming Response
```

---

# Final Recommendation

Do NOT rush into giant model training.

The correct order is:

```text
Phase 1
Retrieval + Citations
        ↓
Phase 2
Embeddings + Semantic Search
        ↓
Phase 3
Small Transformer
        ↓
Phase 4
Clinical Fine-Tuning
        ↓
Phase 5
Advanced Safety + Evaluation
```

For healthcare AI:
- retrieval quality matters more than model size
- citations matter more than fluent writing
- safety matters more than creativity
